In [1]:
import pandas as pd
import numpy as np
import fastf1
import requests

In [2]:
def get_fp1_best(FP1):

    laps = FP1.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds().reset_index()

    return best

In [3]:
def get_fp2_best(FP2):

    laps = FP2.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds().reset_index()

    return best

In [4]:
def get_fp3_best(FP3):

    laps = FP3.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds().reset_index()

    return best

In [5]:
def get_sector_times (FP2):

    laps = FP2.laps

    sector_times = laps.groupby("Driver")[[
    "Sector1Time",
    "Sector2Time",
    "Sector3Time"
    ]].mean().apply(lambda x: x.dt.total_seconds()).reset_index()

    return sector_times

In [6]:
def get_average_laptime (FP2):

    laps = FP2.laps

    avg_lap = laps.groupby("Driver")["LapTime"].mean().dt.total_seconds().reset_index()

    return avg_lap

In [7]:
def get_stint(FP2):

    laps = FP2.laps

    stints = (
        laps.groupby(["Driver","Stint"])
        .size()
        .reset_index(name="LapCount")
    )

    longest_stint = (
        stints.groupby("Driver")["LapCount"]
        .max()
        .reset_index(name="LongestStint")
    )

    return longest_stint

In [8]:
def get_tyre_compound (FP2):

    laps = FP2.laps

    tyres = laps.groupby("Driver")["Compound"].agg(lambda x: x.mode()[0]).reset_index()

    return tyres

In [9]:
def get_qualifying_data(Qualify):

    results = Qualify.results.copy()

    qual_time = results[["Q3", "Q2", "Q1"]].bfill(axis=1).iloc[:, 0]

    qual = results[["Abbreviation"]].copy()
    qual["Qualifying_Time(s)"] = qual_time.dt.total_seconds()

    qual.rename(columns={"Abbreviation": "Driver"}, inplace=True)

    return qual

In [15]:
def get_starting_position(Qualify):

    quali_results = Qualify.results[['Abbreviation', 'Position']]
    quali_results.rename(columns={"Abbreviation":"Driver"},inplace=True)

    return quali_results

In [10]:
def get_race_results(Result):

    race = Result.results[["Abbreviation","Position"]]
    race.rename(columns={"Abbreviation":"Driver"},inplace=True)

    return race

In [ ]:
def Rain(API_KEY,lat,lon,forecast_time):
    url = f"http://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"

    response = requests.get(url)
    data = response.json()

    if "list" not in data:
        print("API ERROR:", data)
    else:

        forecast_data = next(
            (f for f in data["list"] if f["dt_txt"] == forecast_time),
            None
        )
        if forecast_data:
            rain_probability = forecast_data.get("pop", 0)
            temperature = forecast_data["main"]["temp"]
        else:
            forecast_data = data["list"][0]
            rain_probability = forecast_data.get("pop", 0)
            temperature = forecast_data["main"]["temp"]

    return rain_probability,temperature

# Australia GP Data(For Training)

In [16]:
season = 2026
race = 'Australia'

FP1 = fastf1.get_session(season, race, 'FP1')
FP1.load()
FP2 = fastf1.get_session(season, race, 'FP2')
FP2.load()
FP3 = fastf1.get_session(season, race, 'FP3')
FP3.load()

Qualify = fastf1.get_session(season, race, 'Q')
Qualify.load()

Result = fastf1.get_session(season, race, 'R')
Result.load()

print(f"Done Loading Season {season} & Race {race}")

fp1_best = get_fp1_best(FP1)
fp2_best = get_fp2_best(FP2)
fp3_best = get_fp3_best(FP3)
sector_time_best = get_sector_times(FP2)
average_laptime = get_average_laptime(FP2)
stint = get_stint(FP2)
tyre = get_tyre_compound(FP2)
qualifying_time = get_qualifying_data(Qualify)
starting_position = get_starting_position(Qualify)
result = get_race_results(Result)

print("Data Extraction Done.")

fp1_best.set_index("Driver",inplace=True)
fp2_best.set_index("Driver",inplace=True)
fp3_best.set_index("Driver",inplace=True)
sector_time_best.set_index("Driver",inplace=True)
average_laptime.set_index("Driver",inplace=True)
stint.set_index("Driver",inplace=True)
tyre.set_index("Driver",inplace=True)
qualifying_time.set_index("Driver",inplace=True)
starting_position.set_index("Driver",inplace=True)
result.set_index("Driver",inplace=True)

print("Setting Index Done.")

df = fp1_best.copy()
df.rename(columns={"LapTime":"FP1_BestTime(s)"},inplace=True)
df[["FP2_BestTime(s)"]] = fp2_best[["LapTime"]]
df[["FP3_BestTime(s)"]] = fp3_best[["LapTime"]]
df[["Sector1Time(s)","Sector2Time(s)","Sector3Time(s)"]] = sector_time_best[["Sector1Time","Sector2Time","Sector3Time"]]
df[["Average_Laptime(s)"]] = average_laptime[["LapTime"]]
df[["Longest_Stint"]] = stint[["LongestStint"]]
df[["Tyre_Compound"]] = tyre[["Compound"]]
df[["Qualifying_Time(s)"]] = qualifying_time[["Qualifying_Time(s)"]]
df[["Starting_Pos"]] = starting_position[["Position"]]
df[["Race_Result"]] = result[["Position"]]

print("Data Transformation.")

fp1_best.reset_index(inplace=True)
fp2_best.reset_index(inplace=True)
fp3_best.reset_index(inplace=True)
sector_time_best.reset_index(inplace=True)
average_laptime.reset_index(inplace=True)
stint.reset_index(inplace=True)
tyre.reset_index(inplace=True)
qualifying_time.reset_index(inplace=True)
starting_position.reset_index(inplace=True)
result.reset_index(inplace=True)
df.reset_index(inplace=True)

print("Resetting Index Done.")


core           INFO 	Loading data for Australian Grand Prix - Practice 1 [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 14
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 14)
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['1', '3', '5', '6', '10', '11', '12', '14', '16', '18', '23', '27', '30',

Done Loading Season 2026 & Race Australia
Data Extraction Done.
Setting Index Done.
Data Transformation.
Resetting Index Done.


/tmp/ipykernel_881250/2861812837.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  quali_results.rename(columns={"Abbreviation":"Driver"},inplace=True)
/tmp/ipykernel_881250/2097842272.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  race.rename(columns={"Abbreviation":"Driver"},inplace=True)


In [19]:
df.reset_index(inplace=True)

In [20]:
df

,Driver,FP1_BestTime(s),FP2_BestTime(s),FP3_BestTime(s),Sector1Time(s),Sector2Time(s),Sector3Time(s),Average_Laptime(s),Longest_Stint,Tyre_Compound,Qualifying_Time(s),Starting_Pos,Race_Result
0,ALB,83.130,81.847,81.664,36.650031,20.898312,42.942393,100.398929,11,HARD,80.941,15.0,12.0
1,ANT,81.376,79.943,80.324,36.042935,19.788161,41.798556,95.665077,15,HARD,78.811,2.0,2.0
2,BEA,82.682,81.326,80.778,39.039839,20.835774,41.873786,102.072750,15,HARD,80.311,12.0,7.0
3,BOR,81.696,81.668,80.459,39.209964,20.777286,46.708920,99.549182,9,MEDIUM,80.221,10.0,9.0
4,BOT,84.022,83.660,83.514,37.285821,20.556821,43.604600,101.500360,15,MEDIUM,83.244,19.0,19.0
5,COL,83.325,82.619,81.413,38.297846,22.258222,45.171583,101.418318,11,MEDIUM,81.270,16.0,14.0
6,GAS,84.035,82.167,81.071,37.261133,22.742125,41.458429,99.050308,11,MEDIUM,80.501,14.0,10.0
7,HAD,81.087,80.941,80.137,36.938357,23.917321,42.749958,99.028217,13,MEDIUM,79.303,3.0,20.0
8,HAM,80.736,80.050,79.669,36.481677,21.753969,43.648655,99.920000,13,HARD,79.478,7.0,4.0
9,HUL,81.969,81.351,81.067,34.139485,20.528765,41.426839,93.962483,10,MEDIUM,80.303,11.0,22.0


In [ ]:
API_KEY = ''
lat = -25.2744
lon = 133.7751
forecate_time = "2025-03-08 15:00:00"

rain_prob,temp = Rain(API_KEY,lat,lon,forecate_time)

In [24]:
rain_prob

0

In [25]:
temp

19.88